# 4. One-stage detector with anchors: RetinaNet
## How it works
RetinaNet has **no proposal step**. It puts thousands of anchors on the image. For every anchor it says at once: "object of class X" or "background". It also moves the anchor to fit the object.

**Problem:** almost all anchors are background. The model can learn to say "background" every time.

**Solution: focal loss.** Easy background anchors get a very small loss. So the model learns from the hard examples.

## Experiments
* **baseline**: default anchors
* **small anchors**: anchors for our small objects (notebook 01)
* **lower IoU thresholds**: an anchor counts as an object when it overlaps the object by 40% (default 50%). Small objects get more positive anchors.

## Imports and paths
Libraries and the folders for the data, the training files and the figures.

In [1]:
import os
import io
import json
import time
import random
import contextlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw
import torch

In [2]:
if os.path.exists("/kaggle/input"):
    import glob
    DATA = os.path.dirname(glob.glob("/kaggle/input/**/test_images.csv", recursive=True)[0])
    WORK = "/kaggle/working/work"
elif os.name == "nt":
    DATA = "D:/hw2/data"
    WORK = "D:/hw2/work"
else:
    DATA = "/content/data"
    WORK = "/content/work"

# Figures for the report go to the images folder of the repository.
IMAGES = "../images"
RESULTS_FILE = "experiments.csv"
os.makedirs(WORK, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)

CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 36
print("data:", DATA)
print("device:", DEVICE)

data: D:/hw2/data
device: cuda


## Functions
This notebook uses the functions below. Every function does one simple job.

`set_seed` fixes the random numbers. Then a run gives the same result again.

In [3]:
def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

`free_memory` cleans the memory of the GPU and the computer after a model is done.

In [4]:
def free_memory():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

`FolderDataset` reads a prepared folder for torchvision. It gives an image and its boxes. Class numbers start at 1 because 0 is background. Augmentation: a random left-right flip (like in the lecture notebook).

In [5]:
class FolderDataset(torch.utils.data.Dataset):
    """Reads a prepared folder. For torchvision the class numbers start at 1 (0 is background)."""

    def __init__(self, folder):
        self.folder = folder
        self.names = sorted(name[:-4] for name in os.listdir(f"{folder}/images"))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        import torchvision.transforms.functional as TF

        image = Image.open(f"{self.folder}/images/{self.names[i]}.jpg").convert("RGB")
        width, height = image.size
        label_file = f"{self.folder}/labels/{self.names[i]}.txt"
        labels = []
        if os.path.getsize(label_file) > 0:
            labels = np.loadtxt(label_file, ndmin=2)

        boxes = []
        classes = []
        for cls, cx, cy, w, h in labels:
            boxes.append([(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height])
            classes.append(int(cls) + 1)

        image = TF.to_tensor(image)
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        classes = torch.tensor(classes, dtype=torch.int64)

        # Augmentation: random left-right flip.
        if random.random() < 0.5:
            image = image.flip(-1)
            boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
        return image, {"boxes": boxes, "labels": classes}

`collate` puts images of different sizes in one batch.

In [6]:
def collate(batch):
    return tuple(zip(*batch))

`train_torchvision` is a simple training loop (like in the lecture notebook): SGD optimizer, cosine learning rate, batch of 2 images.

In [7]:
def train_torchvision(model, folder, epochs, lr):
    """Simple training loop: SGD with cosine learning rate (from the lecture)."""
    set_seed()
    model = model.to(DEVICE)
    loader = torch.utils.data.DataLoader(FolderDataset(folder), batch_size=2, shuffle=True, collate_fn=collate)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(loader))

    start = time.time()
    for epoch in range(epochs):
        model.train()
        losses = []
        for images, targets in loader:
            images = [image.to(DEVICE) for image in images]
            targets = [{"boxes": t["boxes"].to(DEVICE), "labels": t["labels"].to(DEVICE)} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            # Stop very big steps (keeps training stable).
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}/{epochs}: loss {np.mean(losses):.3f}, time {(time.time() - start) / 60:.1f} min")
    model.eval()
    return model, (time.time() - start) / 60

`torchvision_predictor` turns a torchvision model into a function: image -> boxes, scores, classes.

In [8]:
def torchvision_predictor(model):
    """Returns a function: image -> boxes, scores, classes (0..7)."""
    import torchvision.transforms.functional as TF

    def predict(image):
        with torch.no_grad():
            out = model([TF.to_tensor(image.copy()).to(DEVICE)])[0]
        return out["boxes"].cpu().numpy(), out["scores"].cpu().numpy(), out["labels"].cpu().numpy() - 1

    return predict

`predict_frame` predicts one frame and keeps the 500 best boxes.

In [9]:
def predict_frame(predict, path):
    """Predict one frame. Keep the 500 boxes with the best scores (Kaggle uses at most 500)."""
    image = np.array(Image.open(path).convert("RGB"))
    boxes, scores, classes = predict(image)
    order = np.argsort(-scores)[:500]
    return boxes[order], scores[order], classes[order]

`evaluate` scores a model on the validation frames with the COCO metric (pycocotools). Ignored regions have `iscrowd = 1`, so boxes there are not errors. It also measures the speed (ms per image).

In [10]:
def evaluate(predict, val_frames):
    """
    COCO score on the validation frames.
    Ignored regions are marked iscrowd=1, so boxes inside them are not counted as errors.
    """
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    detections = []
    predictions = []
    start = time.time()
    for row in val_frames.itertuples():
        boxes, scores, classes = predict_frame(predict, row.path)
        predictions.append((boxes, scores, classes))
        for box, score, cls in zip(boxes, scores, classes):
            x1, y1, x2, y2 = [float(v) for v in box]
            detections.append({"image_id": int(row.image_id), "category_id": int(cls) + 1,
                               "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})
    ms_per_image = (time.time() - start) / len(val_frames) * 1000

    with open(f"{DATA}/val/annotations_coco.json") as f:
        ground_truth = json.load(f)
    for a in ground_truth["annotations"]:
        a.setdefault("iscrowd", 0)
        a.setdefault("area", a["bbox"][2] * a["bbox"][3])

    result = {"mAP": 0.0, "AP50": 0.0, "AP_small": 0.0, "AP_medium": 0.0, "AP_large": 0.0}
    if len(detections) > 0:
        # pycocotools prints a lot of text, we hide it.
        with contextlib.redirect_stdout(io.StringIO()):
            gt = COCO()
            gt.dataset = ground_truth
            gt.createIndex()
            E = COCOeval(gt, gt.loadRes(detections), "bbox")
            E.params.imgIds = list(val_frames["image_id"])
            # Up to 500 boxes per frame (scenes are dense).
            E.params.maxDets = [1, 100, 500]
            E.evaluate()
            E.accumulate()
            E.summarize()
        result = {"mAP": E.stats[0], "AP50": E.stats[1], "AP_small": E.stats[3],
                  "AP_medium": E.stats[4], "AP_large": E.stats[5]}
    for key in result:
        result[key] = round(float(result[key]), 4)
    result["ms/img"] = round(ms_per_image, 1)
    return result, predictions

`save_result` adds one line to `experiments.csv`. All notebooks write to this file. We use it in the report.

In [11]:
def save_result(notebook, model, experiment, metrics, minutes):
    """Add one line to experiments.csv. All notebooks write to the same file."""
    row = {"notebook": notebook, "model": model, "experiment": experiment}
    row.update(metrics)
    row["train_min"] = round(minutes, 1)
    if os.path.exists(RESULTS_FILE):
        table = pd.read_csv(RESULTS_FILE, dtype={"notebook": str})
        table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    else:
        table = pd.DataFrame([row])
    table.to_csv(RESULTS_FILE, index=False)

## Settings
All models in notebooks 03-06 use the same settings. So we can compare them in a fair way.

The settings are small. One experiment takes only a few minutes.

In [12]:
SIZE = 640
EPOCHS = 3
TRAIN_FOLDER = f"{WORK}/fast/train"
DATA_YAML = f"{WORK}/fast/data.yaml"
val_frames = pd.read_csv(f"{WORK}/val_frames.csv")

print("training images:", len(os.listdir(f"{TRAIN_FOLDER}/images")))
print("validation frames:", len(val_frames))

training images: 271
validation frames: 146


## Build the model
* `anchor_sizes`: base anchor size on each level. Every level has 3 sizes and 3 shapes, so 9 anchors per position.
* `fg_iou`, `bg_iou`: overlap above `fg_iou` = object, below `bg_iou` = background.

We change the class head to 9 outputs.

In [13]:
from functools import partial
from torch import nn
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator


def build_retinanet(anchor_sizes, fg_iou, bg_iou):
    model = retinanet_resnet50_fpn_v2(
        weights="DEFAULT",
        min_size=SIZE,
        max_size=SIZE,
        fg_iou_thresh=fg_iou,
        bg_iou_thresh=bg_iou,
        detections_per_img=500,
        score_thresh=0.001,
    )
    sizes = tuple((s, int(s * 1.26), int(s * 1.59)) for s in anchor_sizes)
    shapes = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    model.anchor_generator = AnchorGenerator(sizes, shapes)
    model.head.classification_head = RetinaNetClassificationHead(256, 9, 9, norm_layer=partial(nn.GroupNorm, 32))
    return model

## Train and score
Same steps as in notebook 03.

In [14]:
experiments = {
    "baseline": {"anchor_sizes": (32, 64, 128, 256, 512), "fg_iou": 0.5, "bg_iou": 0.4},
    "small anchors": {"anchor_sizes": (16, 32, 64, 128, 256), "fg_iou": 0.5, "bg_iou": 0.4},
    "small anchors + lower IoU": {"anchor_sizes": (16, 32, 64, 128, 256), "fg_iou": 0.4, "bg_iou": 0.3},
}

results = []
for name in experiments:
    print("=====", name)
    model = build_retinanet(**experiments[name])
    model, minutes = train_torchvision(model, TRAIN_FOLDER, EPOCHS, lr=0.0025)
    metrics, _ = evaluate(torchvision_predictor(model), val_frames)
    save_result("04", "RetinaNet", name, metrics, minutes)
    results.append({"experiment": name, **metrics, "train_min": round(minutes, 1)})
    del model
    free_memory()

pd.DataFrame(results)

===== baseline
epoch 1/3: loss 1.103, time 0.6 min
epoch 2/3: loss 0.887, time 1.2 min
epoch 3/3: loss 0.823, time 1.7 min
===== small anchors
epoch 1/3: loss 1.131, time 0.6 min
epoch 2/3: loss 0.874, time 1.2 min
epoch 3/3: loss 0.809, time 1.8 min
===== small anchors + lower IoU
epoch 1/3: loss 1.126, time 0.6 min
epoch 2/3: loss 0.853, time 1.2 min
epoch 3/3: loss 0.789, time 1.8 min


,experiment,mAP,AP50,AP_small,AP_medium,AP_large,ms/img,train_min
0,baseline,0.0383,0.0831,0.0120,0.0264,0.1079,168.6,1.7
1,small anchors,0.0264,0.0848,0.0111,0.0228,0.0747,196.8,1.8
2,small anchors + lower IoU,0.0251,0.0779,0.0107,0.0207,0.0722,175.3,1.8


## Compare with Faster R-CNN
We read all results of notebooks 03 and 04 from `experiments.csv`.

In [15]:
table = pd.read_csv(RESULTS_FILE, dtype={"notebook": str})
table = table[table["notebook"].isin(["03", "04"])]
table[["model", "experiment", "mAP", "AP50", "AP_small", "ms/img", "train_min"]]

,model,experiment,mAP,AP50,AP_small,ms/img,train_min
0,Faster R-CNN,baseline,0.0885,0.2047,0.0481,238.6,3.0
1,Faster R-CNN,small anchors,0.0961,0.2190,0.0416,268.0,3.0
2,Faster R-CNN,small anchors + more proposals,0.0977,0.2282,0.0388,321.2,3.0
3,RetinaNet,baseline,0.0387,0.0863,0.0140,216.3,2.0
4,RetinaNet,small anchors,0.0275,0.0897,0.0108,215.0,2.1
5,RetinaNet,small anchors + lower IoU,0.0263,0.0791,0.0120,209.0,1.9
11,Faster R-CNN,baseline,0.0953,0.2232,0.0417,212.8,2.8
12,Faster R-CNN,small anchors,0.0950,0.2204,0.0419,218.1,2.8
13,Faster R-CNN,small anchors + more proposals,0.0915,0.2167,0.0437,297.9,2.9
14,RetinaNet,baseline,0.0383,0.0831,0.0120,168.6,1.7


## Conclusions

| experiment | mAP | AP50 | AP_small | AP_medium | AP_large | ms/img |
|---|---|---|---|---|---|---|
| baseline | **0.0383** | 0.0831 | **0.0120** | **0.0264** | **0.1079** | **168.6** |
| small anchors | 0.0264 | **0.0848** | 0.0111 | 0.0228 | 0.0747 | 196.8 |
| small anchors + lower IoU | 0.0251 | 0.0779 | 0.0107 | 0.0207 | 0.0722 | 175.3 |
| *best Faster R-CNN (baseline)* | *0.0953* | *0.2232* | *0.0417* | *0.0754* | *0.2266* | *212.8* |

* RetinaNet is **about 2.5 times worse** than Faster R-CNN after the same 3 epochs. It is only a little faster (169 vs 213 ms per image).
* **Small anchors** made RetinaNet worse. The pretrained head learned the default anchor sizes on COCO. With new sizes it must learn again, and 3 epochs are not enough.
* **Lower IoU thresholds** made it a bit worse again. More anchors become "object", but many of them cover the object only a little. These are bad training examples.
* The same settings run two times give different scores (Faster R-CNN baseline: 0.0885 and 0.0953). So differences smaller than about 0.01 mAP are noise.

**Problems without the proposal stage**
* RetinaNet checks every anchor on the image: about 100 000 anchors at 640 px. Almost all of them are background. Faster R-CNN gives only about 2000 proposals to its second stage, so the second stage sees a much better balance.
* There is no second look at the box, so the boxes are less exact. Even AP50 is low (0.08 vs 0.22).

**How to fix it**
* **Focal loss** (already in RetinaNet) gives a small loss to easy background anchors. So the model learns from hard examples.
* **Longer training:** one-stage models with focal loss usually need more epochs than 3.
* **Better assignment of anchors**, for example ATSS: it picks positive anchors by the statistics of each object, not by one fixed IoU threshold.
* **Bigger input or tiles:** our objects are small, so they cover very few anchors at 640 px.